# A100 Gaussian training ablation

This is a **diagnostic notebook**, not another full-quality splat run. It requires the passing CPU cache audit and final pre-training cache for the same input. Drive inputs are restored and verified **once** into the session's local disk. Six deterministic 5K training variants then compare legacy control, dense seeds, masks, depth, adaptive density, and the full learned combination. Pairwise 2.5K variants run only when needed. The notebook publishes `<input>_training_ablation` with reports and contact sheets; it deliberately publishes no PLY and no viewer.

In [ ]:
INPUT_FOLDER = ""  # @param {type:"string"}


In [ ]:
import json, shutil, subprocess
gpu_line = subprocess.run([
    'nvidia-smi', '--query-gpu=name,memory.total',
    '--format=csv,noheader,nounits',
], check=True, capture_output=True, text=True).stdout.splitlines()[0]
gpu_name, memory_mib = (part.strip() for part in gpu_line.rsplit(',', 1))
vram_gib = float(memory_mib) / 1024.0
assert 'A100' in gpu_name.upper(), f'A100 required; detected {gpu_name}'
assert vram_gib >= 75.0, f'At least 75 GiB VRAM required; detected {vram_gib:.1f}'
disk_gib = shutil.disk_usage('/content').free / (1024 ** 3)
assert disk_gib >= 80.0, f'At least 80 GiB local disk required; detected {disk_gib:.1f}'
print(json.dumps({'gpu': gpu_name, 'vram_gib': round(vram_gib, 1), 'disk_free_gib': round(disk_gib, 1)}, sort_keys=True))


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive').resolve()


In [ ]:
import json, unicodedata
from pathlib import Path, PurePosixPath
raw_folder = INPUT_FOLDER.strip()
if not raw_folder:
    raw_folder = input('MyDrive-relative input folder: ').strip()
assert raw_folder and '\\' not in raw_folder, 'Use a MyDrive-relative POSIX path'
assert not any(unicodedata.category(ch) == 'Cc' for ch in raw_folder)
folder = PurePosixPath(raw_folder)
assert not folder.is_absolute() and folder.parts
assert all(part not in {'', '.', '..'} for part in folder.parts)
assert not raw_folder.endswith(('_result', '_training_ablation', '_learned_test_result', '_learned_test_diagnostics', '_learned_test_cache'))
INPUT_PATH = DRIVE_ROOT.joinpath(*folder.parts).resolve()
INPUT_PATH.relative_to(DRIVE_ROOT)
assert INPUT_PATH.is_dir(), f'Input folder does not exist: {INPUT_PATH}'
CACHE_PATH = INPUT_PATH.with_name(INPUT_PATH.name + '_learned_test_cache')
RESULT_PATH = INPUT_PATH.with_name(INPUT_PATH.name + '_training_ablation')
RUN_SPEC = {'schema_version': 1, 'input_folder': folder.as_posix(), 'publish': {'replace_owned_result': True}}
SPEC_PATH = Path('/content/learned_ablation_spec.json')
with SPEC_PATH.open('w', encoding='utf-8') as handle:
    json.dump(RUN_SPEC, handle, sort_keys=True, separators=(',', ':'))
print(f'Input: {INPUT_PATH}')
print(f'Verified cache: {CACHE_PATH}')
print(f'Diagnostic result: {RESULT_PATH}')


In [ ]:
import shutil, subprocess
from pathlib import Path
SOURCE_ROOT = Path('/content/gaussian-splatter-src')
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
REPOSITORY_URL = 'https://github.com/mehmettahacumurcu/gaussian-splatter.git'
COMMIT_SHA = '3d5e39f27afbcf095d6dde4881f68398dfaa44e2'
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(SOURCE_ROOT)], check=True)
subprocess.run(['git', '-C', str(SOURCE_ROOT), 'checkout', '--detach', COMMIT_SHA], check=True)
actual = subprocess.run(['git', '-C', str(SOURCE_ROOT), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
assert actual == COMMIT_SHA, 'Immutable source checkout mismatch'


In [ ]:
import subprocess
subprocess.run(['bash', 'colab/static_notebook_bootstrap.sh'], cwd=SOURCE_ROOT, check=True)


In [ ]:
import sys
sys.path.insert(0, str(SOURCE_ROOT))
from huggingface_hub import snapshot_download
from experiments.learned_quality.dependencies import (
    install_learned_environment, materialize_pinned_assets,
)
def resolved_revision(local_path):
    metadata_root = Path(local_path) / '.cache' / 'huggingface' / 'download'
    revisions = set()
    for metadata in metadata_root.rglob('*.metadata'):
        first = metadata.read_text(encoding='utf-8').splitlines()[0].strip()
        if len(first) == 40:
            revisions.add(first)
    assert len(revisions) == 1, f'Cannot authenticate checkpoint: {local_path}'
    return revisions.pop()
LEARNED_ENV = install_learned_environment(Path(sys.executable).resolve())
LEARNED_ASSETS = materialize_pinned_assets(
    LEARNED_ENV, downloader=snapshot_download, resolve_revision=resolved_revision,
)


In [ ]:
from experiments.learned_quality.dependencies import verify_learned_environment
MODEL_MANIFEST = verify_learned_environment(LEARNED_ENV, LEARNED_ASSETS)
assert MODEL_MANIFEST.path == Path('/content/learned-env/model_manifest.json')
print(f'Verified model manifest: {MODEL_MANIFEST.path}')


In [ ]:
import json, os, subprocess, traceback
from pathlib import Path
from google.colab import drive, runtime
environment = dict(os.environ)
environment['LEARNED_MODEL_MANIFEST'] = str(MODEL_MANIFEST.path)
environment['PYTHONUNBUFFERED'] = '1'
failure = None
try:
    completed = subprocess.run(
        [
            "/content/learned-env/bin/python",
            "-u", "-m", "scripts.learned_quality_ablation_run",
            "--spec", "/content/learned_ablation_spec.json",
            "--source-revision", COMMIT_SHA,
            "--model-manifest", str(MODEL_MANIFEST.path),
        ],
        cwd=SOURCE_ROOT, env=environment, check=False,
    )
    receipt_path = Path('/content/learned_ablation_result.json')
    if not receipt_path.is_file():
        raise RuntimeError('Ablation ended without a run receipt')
    receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
    if completed.returncode != 0 or receipt.get('status') != 'success':
        raise RuntimeError(f'Training ablation failed or is partial: {receipt}')
    actual = Path(receipt['final_path']).resolve()
    if actual != RESULT_PATH.resolve():
        raise RuntimeError(f'Unexpected ablation result path: {actual}')
    success = json.loads((actual / '_SUCCESS.json').read_text(encoding='utf-8'))
    if success.get('run_id') != receipt.get('run_id'):
        raise RuntimeError('Ablation success marker does not match this run')
    required = (
        'ablation_report.json', 'ablation_summary.md', 'metrics.csv',
        'psnr_plot.png', 'environment.json', 'staging_manifest.json',
    )
    missing = [name for name in required if not (actual / name).is_file()]
    if missing:
        raise RuntimeError(f'Published ablation report is incomplete: {missing}')
    print(f'Ablation report: {actual}')
    print(f'Diagnosis: {receipt.get("diagnosis")}')
    print('Diagnostic matrix and Drive publication completed successfully.')
except BaseException as exc:
    failure = exc
    print(f'Run ended with {type(exc).__name__}: {exc}')
    traceback.print_exception(type(exc), exc, exc.__traceback__)
finally:
    print('Flushing outstanding Google Drive writes...')
    try:
        drive.flush_and_unmount()
    except BaseException as flush_error:
        print(f'Drive flush/unmount failed: {flush_error}')
    print('Releasing the Colab runtime now.')
    try:
        runtime.unassign()
    except BaseException as release_error:
        print(f'Runtime release request failed: {release_error}')
        if failure is None:
            failure = release_error
if failure is not None:
    raise failure
